# 📈 Crypto Pulse - POC Dashboard

This dashboard provides a unified view of market data, sentiment analysis, and news feed.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
import os
import glob
from datetime import datetime

import plotly.io as pio
pio.templates.default = "plotly_dark"

def load_kline_data(symbol):
    file_path = f'../data/historical/{symbol.lower()}usdt_raw_klines.json'
    if not os.path.exists(file_path): return None
    with open(file_path, 'r') as f: data = json.load(f)
    df = pd.DataFrame(data, columns=['open_time', 'open', 'high', 'low', 'close', 'volume', 'ct', 'qav', 'nt', 'tbb', 'tbq', 'i'])
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    df[['open', 'high', 'low', 'close', 'volume']] = df[['open', 'high', 'low', 'close', 'volume']].apply(pd.to_numeric)
    return df

## 2. Market Overview & Performance Comparison

In [ ]:
def show_dashboard():
    # 1. Main Charts
    symbols = ['btc', 'eth', 'bnb', 'sol', 'ada']
    all_prices = {}
    
    fig = make_subplots(rows=2, cols=1, vertical_spacing=0.1, subplot_titles=('BTC/USDT Candlestick', 'Relative Assets Performance (%)'))
    
    # BTC Candlestick
    btc_df = load_kline_data('btc')
    if btc_df is not None:
        fig.add_trace(go.Candlestick(x=btc_df['open_time'],
                                    open=btc_df['open'], high=btc_df['high'],
                                    low=btc_df['low'], close=btc_df['close'], name='BTC Price'), row=1, col=1)
    
    # Performance Comparison (Normalized to 100)
    for s in symbols:
        df = load_kline_data(s)
        if df is not None:
            normalized = (df['close'] / df['close'].iloc[0]) * 100
            fig.add_trace(go.Scatter(x=df['open_time'], y=normalized, name=f'{s.upper()} Performance'), row=2, col=1)
            
    fig.update_layout(height=900, title='CryptoPulse POC Dashboard', template='plotly_dark', xaxis_rangeslider_visible=False)
    fig.show()

show_dashboard()

## 3. Sentiment Indicator

In [ ]:
fig = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = 65, # Placeholder
    title = {'text': "Market Sentiment Index (Greed)"},
    gauge = {'axis': {'range': [0, 100]}, 'bar': {'color': "green"}, 
             'steps': [{'range': [0, 40], 'color': "red"}, {'range': [40, 60], 'color': "gray"}, {'range': [60, 100], 'color': "green"}]}
))
fig.update_layout(template='plotly_dark', height=400)
fig.show()